# Notebook 02 — Preprocessing

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Inputs:** `data/raw/nasa.csv`  
**Outputs:** `output/dataset_full.npz`, `output/feature_names.json`, `output/scaler.pkl`  
**Purpose:** Apply column drops decided in EDA, encode target, stratified train/test split,
StandardScaler. Produces the canonical dataset consumed by all subsequent notebooks.

Every random operation uses `random_state=42`.

---

## 1. Load raw CSV and apply column drops

In [ ]:
import pandas as pd
import numpy as np
import json, pickle, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

os.makedirs('output', exist_ok=True)

df = pd.read_csv('data/raw/nasa.csv')

# Drop list established in 01_eda.ipynb
ALWAYS_DROP = [
    'Name', 'Close Approach Date', 'Orbit Determination Date',
    'Equinox', 'Orbiting Body',
]
UNIT_REDUNDANT = [
    'Est Dia in M(min)', 'Est Dia in M(max)',
    'Est Dia in Miles(min)', 'Est Dia in Miles(max)',
    'Est Dia in Feet(min)', 'Est Dia in Feet(max)',
    'Relative Velocity km per hr', 'Miles per hour',
    'Miss Dist.(lunar)', 'Miss Dist.(kilometers)', 'Miss Dist.(miles)',
]
DERIVED_OR_ADMIN = [
    'Orbital Period', 'Mean Motion', 'Perihelion Time',
    'Epoch Osculation', 'Epoch Date Close Approach', 'Orbit ID',
]
ORIENTATION_ANGLES = ['Asc Node Longitude', 'Perihelion Arg', 'Mean Anomaly']

ALL_DROP = ALWAYS_DROP + UNIT_REDUNDANT + DERIVED_OR_ADMIN + ORIENTATION_ANGLES

# Extract IDs before dropping — needed by notebook 07 for game export
neo_ids_all = df['Neo Reference ID'].values

df_clean = df.drop(columns=ALL_DROP + ['Neo Reference ID'])

print(f"Shape after drops: {df_clean.shape}")
print(f"Columns retained: {list(df_clean.columns)}")
print(f"\nNull counts:")
print(df_clean.isnull().sum().to_string())

## 2. Encode target

`Hazardous` is a Python bool column. Convert to int (False→0, True→1) for sklearn.

In [ ]:
df_clean['Hazardous'] = df_clean['Hazardous'].astype(int)
X = df_clean.drop(columns=['Hazardous'])
y = df_clean['Hazardous']

print(f"X shape: {X.shape}")
print(f"Feature names ({len(X.columns)}):")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\ny — SAFE (0): {(y==0).sum():,}  |  HAZARDOUS (1): {(y==1).sum():,}")

## 3. Stratified train/test split (80/20)

Stratification ensures both splits reflect the original class ratio.
We also split the `neo_ids_all` array in parallel so notebook 07 can trace
every training/test row back to its original NASA catalogue identifier.

In [ ]:
X_arr = X.values
y_arr = y.values

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X_arr, y_arr, neo_ids_all,
    test_size=0.2, random_state=42, stratify=y_arr,
)

print(f"Train: {X_train.shape[0]:,} rows — hazardous: {y_train.sum():,} ({100 * y_train.mean():.2f}%)")
print(f"Test:  {X_test.shape[0]:,}  rows — hazardous: {y_test.sum():,} ({100 * y_test.mean():.2f}%)")
print(f"\nStratification check — original: {100 * y_arr.mean():.2f}%  |  train: {100 * y_train.mean():.2f}%  |  test: {100 * y_test.mean():.2f}%")

## 4. StandardScaler

Fit **only on the training set** to avoid data leakage.
The fitted scaler is applied to both train and test, and saved for use in all
downstream notebooks (03–07).

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Fitted on {X_train.shape[0]:,} training samples")
print(f"\nFeature means (training):")
for name, mean, std in zip(X.columns, scaler.mean_, scaler.scale_):
    print(f"  {name:<35}  mean={mean:>12.4f}   std={std:>10.4f}")

## 5. Save artifacts

Three files produced here are the source of truth for all downstream notebooks:
- `dataset_full.npz` — scaled X/y arrays + Neo IDs for train and test sets
- `feature_names.json` — column order for indexing (critical for MOID ablation in nb 05)
- `scaler.pkl` — fitted scaler for inverse-transforming or re-scaling new data

In [ ]:
feature_names = list(X.columns)

np.savez(
    'output/dataset_full.npz',
    X_train=X_train_sc, X_test=X_test_sc,
    y_train=y_train, y_test=y_test,
    ids_train=ids_train, ids_test=ids_test,
)
print("✓ output/dataset_full.npz")

with open('output/feature_names.json', 'w') as f:
    json.dump(feature_names, f, indent=2)
print("✓ output/feature_names.json")

with open('output/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ output/scaler.pkl")

# ── Round-trip verification ───────────────────────────────────────────────────
check = np.load('output/dataset_full.npz', allow_pickle=True)
print(f"\nVerification:")
print(f"  X_train  : {check['X_train'].shape}")
print(f"  X_test   : {check['X_test'].shape}")
print(f"  y_train  : {check['y_train'].shape}   hazardous={int(check['y_train'].sum())}")
print(f"  y_test   : {check['y_test'].shape}    hazardous={int(check['y_test'].sum())}")
print(f"  ids_train: {check['ids_train'].shape}")
print(f"  ids_test : {check['ids_test'].shape}")
print(f"\nPhase 1 complete — dataset_full.npz ready for notebook 03.")